# 02 · Balanced sampling, Braintrust upload & queuing a run

This notebook builds a **deterministic, class-balanced dataset slice** exactly the way
the slice builders do (`scripts/braintrust/create_braintrust_800_dataset.py`):

1. Sample `N` images per class with a seeded `random.Random`.
2. De-duplicate in **rendered-pixel space** (hash the normalized PNG, never raw bytes).
3. Normalize each image to a grayscale 1024x1024 PNG and upload it to Braintrust as a
   row attachment.
4. Queue an eval run against the new slice.


## 0. Bootstrap: repo path + credentials

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


In [ ]:
from src.braintrust_config import load_braintrust_config
from src.env_utils import require_env

config = load_braintrust_config()      # braintrust.env first, then .env
api_key = require_env("OPENROUTER_API_KEY")[0]

print("project:", config.project_name)
print("project_id:", config.project_id)
print("dataset:", config.dataset_project, "/", config.dataset)
print("model:", config.model)
print("braintrust api_key set:", bool(config.api_key))
print("openrouter api_key set:", bool(api_key))


## 1. Sampling parameters

`N_PER_CLASS` and `SEED` are your spec: the same seed + source always reproduces the
same slice. The slice has `16 * N_PER_CLASS` rows (each of the 16 classes in
`src.constants.DOCUMENT_CLASSES`).

In [ ]:
from src.constants import DOCUMENT_CLASSES

N_PER_CLASS = 2   # EDIT: images per class -> 16 * N_PER_CLASS total rows
SEED = 42         # EDIT: deterministic sampling seed
print(f"{N_PER_CLASS} per class x {len(DOCUMENT_CLASSES)} classes = {N_PER_CLASS * len(DOCUMENT_CLASSES)} rows")


## 2. Randomized balanced sample from a local tree

This mirrors `scripts/datasets/create_balanced_dataset.py`: walk one directory per
class and draw `N_PER_CLASS` files with a seeded RNG. (For larger slices from a Hugging
Face parquet mirror, the 800/1600 builders stream the parquet and apply the same
sampling + dedup logic.)

In [ ]:
import random
from pathlib import Path

from src.image_utils import find_images

source_root = ROOT / "rvlcdip_dataset"   # EDIT: local RVL-CDIP per-class directory tree

rng = random.Random(SEED)


def sample_balanced(source_root: Path, n_per_class: int, rng: random.Random):
    selected: list[tuple[str, Path]] = []
    for cls in DOCUMENT_CLASSES:
        files = find_images(source_root / cls, recursive=True)
        if len(files) < n_per_class:
            print(f"WARNING: {cls}: only {len(files)} files found")
        for path in rng.sample(files, min(n_per_class, len(files))):
            selected.append((cls, path))
    return selected


selected = sample_balanced(source_root, N_PER_CLASS, rng)
print(f"{len(selected)} images sampled ({N_PER_CLASS}/class x {len(DOCUMENT_CLASSES)} classes)")
for cls, path in selected[:6]:
    print(" ", cls, path.name)


## 3. Normalize + pixel-hash de-duplication

De-duplication is enforced on the **normalized rendered PNG**, so identical images from
different files cannot slip past. Each image is rendered to a grayscale 1024x1024 PNG
first, hashed, and skipped if the hash was already accepted.

In [ ]:
import hashlib
from io import BytesIO

from PIL import Image

from src.image_utils import resize_with_padding


def to_png_bytes(img: Image.Image, target_size=(1024, 1024)) -> bytes:
    img = img.convert("L")
    img = resize_with_padding(img, target_size, fill=255)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def pixel_hash(png_bytes: bytes) -> str:
    return hashlib.sha256(png_bytes).hexdigest()


cls, path = selected[0]
png = to_png_bytes(Image.open(path))
print(cls, path.name, "->", len(png), "bytes, hash", pixel_hash(png)[:16])


## 4. Upload to Braintrust (idempotent)

The upload matches the documented pattern used by every slice builder:
`braintrust.login` -> `init_dataset` -> `insert` with an `Attachment` payload -> `flush`/`close`. An existing dataset with the same name is deleted first so re-runs
are safe. Each row records its `expected` label and provenance metadata.

In [ ]:
import braintrust
from braintrust import Attachment

from src.braintrust_utils import delete_dataset_by_name

dataset_name = f"notebook_balanced_{N_PER_CLASS}"

braintrust.login(api_key=config.api_key)
deleted = delete_dataset_by_name(config.api_key, config.project_id, dataset_name, config.api_base)
print("deleted existing dataset" if deleted else "no existing dataset to delete")

dataset = braintrust.init_dataset(project_id=config.project_id, name=dataset_name)
used: set[str] = set()
i = 0
for cls, path in selected:
    png = to_png_bytes(Image.open(path))
    h = pixel_hash(png)
    if h in used:
        print("  skip pixel duplicate:", path.name)
        continue
    used.add(h)
    i += 1
    fn = f"rvl_cdip__{cls}__{i:04d}.png"
    dataset.insert(
        input={
            "image": Attachment(data=png, filename=fn, content_type="image/png"),
            "metadata": {"class": cls, "placeholder": False},
        },
        expected=cls,
        metadata={"source": "notebook-balanced-sample", "seed": SEED, "n_per_class": N_PER_CLASS},
    )
dataset.flush()
dataset.close()
print(f"Uploaded {len(used)} unique images -> dataset '{dataset_name}'")


## 5. Queue an eval run against the slice

Preflight first (validates prompt + dataset, **spends no credits**), then the eval
runner (`braintrust_openrouter_input.py`) executes the slice and streams results into
both Braintrust and a local JSONL manifest. This step spends OpenRouter credits.

In [ ]:
import subprocess
import sys


def run_script(rel_script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / rel_script), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)


experiment_name = f"notebook_{config.model.replace('/', '_')}_v17.2_{dataset_name}"

# 0) Preflight: validates prompt + dataset, spends no credits.
run_script("braintrust/preflight_eval.py", "--dataset", dataset_name, "--prompt-version", "v17.2")

# 1) Queue the eval run (this spends OpenRouter credits).
run_script(
    "braintrust/braintrust_openrouter_input.py",
    "--dataset", dataset_name,
    "--prompt-version", "v17.2",
    "--model", config.model,
    "--experiment-name", experiment_name,
    "--manifest", str(ROOT / "reports" / "manifests" / f"{experiment_name}.jsonl"),
)


## Next

Notebook **03 · Watchers, evaluators & full experiment launch** covers preflight,
monitoring a run from the manifest, crash-proof resume, and the post-run scoring/
reporting chain.